In [ ]:
from tabular_datasets_utils import dataset_to_numpy, load_dutch
import random
import numpy as np
from torch.utils.data import Dataset
from opacus.utils.batch_memory_manager import BatchMemoryManager
import torch
import os
import torch.nn as nn
from torch import nn
from opacus import PrivacyEngine
import torch.optim as optim
import torch
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
import tqdm

In [ ]:
def seed_everything():
    seed = 42
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True

In [3]:
seed_everything()

# Prepare the dataset

In [4]:
class TabularDataset(Dataset):
    def __init__(self, x, z, y):
        """
        Initialize the custom dataset with x (features), z (sensitive values), and y (targets).

        Args:
        x (list of tensors): List of input feature tensors.
        z (list): List of sensitive values.
        y (list): List of target values.
        """
        self.samples = x
        self.sensitive_features = z
        self.targets = y
        self.indexes = range(len(self.samples))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        """
        Get a single data point from the dataset.

        Args:
        idx (int): Index to retrieve the data point.

        Returns:
        sample (dict): A dictionary containing 'x', 'z', and 'y'.
        """
        x_sample = self.samples[idx]
        z_sample = self.sensitive_features[idx]
        y_sample = self.targets[idx]

        return x_sample, z_sample, y_sample, self.indexes[idx], idx

In [5]:
def prepare_dutch(base_path):
    tmp = load_dutch(dataset_path=base_path)
    tmp = dataset_to_numpy(*tmp, num_sensitive_features=1)

    x = tmp[0]
    y = tmp[2]
    z = tmp[1]

    xyz = list(zip(x, y, z))
    random.shuffle(xyz)
    x, y, z = zip(*xyz)
    train_size = int(len(y) * 0.8)

    x_train = np.array(x[:train_size])
    x_test = np.array(x[train_size:])
    y_train = np.array(y[:train_size])
    y_test = np.array(y[train_size:])
    z_train = np.array(z[:train_size])
    z_test = np.array(z[train_size:])

    train_dataset = TabularDataset(
        x=np.hstack((x_train, np.ones((x_train.shape[0], 1)))).astype(np.float32),
        z=z_train.astype(np.float32),
        y=y_train.astype(np.float32),
    )

    test_dataset = TabularDataset(
        x=np.hstack((x_test, np.ones((x_test.shape[0], 1)))).astype(np.float32),
        z=z_test.astype(np.float32),
        y=y_test.astype(np.float32),
    )

    return train_dataset, test_dataset

In [6]:
dutch_train, dutch_test = prepare_dutch("/raid/lcorbucci/data/dutch/")

Using ['sex_binary'] as sensitive feature(s).


# Disparity Loss

In [7]:
class DisparityRegularizationLoss(nn.Module):
    """This class defines the regularization loss as proposed in
    https://arxiv.org/abs/2302.09183.
    It uses the definition of demographic parity to compute the
    fairness violation term for each batch and then it uses this
    violation term as a regularization term to add to the loss.
    """

    def __init__(self, weight=None, size_average=True, estimation=0.5) -> None:
        """Initialization of the regularization loss."""
        super().__init__()
        self.estimation = estimation

    def forward(
        self,
        sensitive_attribute_list: torch.tensor,
        device: torch.device,
        predictions: torch.tensor,
        possible_sensitive_attributes: list,
        possible_targets: list,
        average_probabilities: dict = None,
        wandb_run=None,
        batch=None,
        global_computation=False,
        json_file=None,
    ) -> torch.tensor:
        """This function computes the regularization term.
        It takes as input the sensitive attribute list, the targets,
        the device and the predictions computed with the model.
        It returns the regularization term.

        What we do here:
        - We compute the softmax of the predictions
        - Then we consider the possible combinations of targets and sensitive features
            and we compute the corresponding fairness violation term
        - We return the maximum violation term among all the possible combinations

        Args:
            sensitive_attribute_list (np.array): a list with the value of
                the sensitive attribute for each sample in the batch
            device (str): the device we're using to train the model
            predictions (np.array): the predictions of the model for the batch of data
            possible_sensitive_attributes (list): the possible values of the sensitive
                attribute
            possible_targets (list): the possible target values we have in this
                dataset
            average_probabilities (dict): in case of Federated learning, if a client
                has only a subset of the possible sensitive attributes, we can use the
                average probabilities of the other clients to estimate the probabilities
                of the missing sensitive attributes. This is None in centralised learning

        Example:
            >>> sensitive_attribute_list = torch.tensor([1, 1, -1, -1, 1, -1])
            >>> device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            >>> predictions = torch.tensor([[0.1, 0.9], [0.2, 0.8], [0.3, 0.7],
                [0.4, 0.6], [0.5, 0.5], [0.6, 0.4]])
            >>> possible_sensitive_attributes = [1, -1]
            >>> possible_targets = [0, 1]
            >>> regularization_loss = RegularizationLoss()
            >>> regularization_loss(sensitive_attribute_list, device, predictions,
                possible_sensitive_attributes, possible_targets)

        Returns:
            float: the disparity metric computed on the data passed as parameter
        """

        fairness_violations = []
        # We compute the softmax of the predictions. We do this because
        # we can't use the argmax function on the nn output,
        # because we need differentiable results
        softmax_ = F.softmax(predictions, dim=1)

        # convert the list of sensitive attributes to a tensor and move it to the device
        sensitive_attribute_list = torch.tensor([int(item) for item in sensitive_attribute_list])
        sensitive_attribute_list = sensitive_attribute_list.to(device)

        # We compute the argmax of the predictions, this is used to count
        # the number of samples for each class that are predicted with one class
        # or with the other.
        predictions_argmax = torch.argmax(torch.tensor(predictions), dim=1).to(device)
        # we convert the possible targets and the possible sensitive attributes to a list
        # just to be sure that the values are integers
        possible_targets = [int(item) for item in possible_targets]
        possible_sensitive_attributes = [int(item) for item in possible_sensitive_attributes]

        global_counters = {}

        for target in possible_targets:
            for z in possible_sensitive_attributes:
                # Z_eq_z and Z_not_eq_z are the denominators that we will use
                # in the DPL formula. |Z=z| and |Z!=z|
                Z_eq_z = len(sensitive_attribute_list[sensitive_attribute_list == z])
                Z_not_eq_z = len(sensitive_attribute_list[sensitive_attribute_list != z])

                # We get the number of samples that are predicted with the target class
                # target and that have the sensitive attribute equal to z:  |Y = k, Z = z|.
                # In this case we just sum the columns of the rows that
                # respect the previous constraint.
                # Example: Given [[0.2, 0.8], [0.4, 0.6], [0.3, 0.7]], suppose
                # that to compute Y_eq_k_and_Z_eq_z we have to consider only
                # the first and the third row and that we are considering the class 1.
                # In this case we will sum 0.8 and 0.7.
                Y_eq_k_and_Z_eq_z = torch.sum(
                    softmax_[(predictions_argmax == target) & (sensitive_attribute_list == z)][:, target]
                )

                # Here we compute |Y = k, Z != z| with the same strategy we used to
                # compute |Y = k, Z = z|.
                Y_eq_k_and_Z_not_eq_z = torch.sum(
                    softmax_[(predictions_argmax == target) & (sensitive_attribute_list != z)][:, target]
                )

                global_counters[f"{target}|{z}"] = Y_eq_k_and_Z_eq_z

                # Now we can compute the violation term that we will
                # sum to our loss. We have to consider the case in which
                # Z_eq_z or Z_not_eq_z are equal to 0, because in this case
                # we will have a division by 0.
                # If this doesn't happen we can compute the violation term
                # using the classic formula |P(Y=y|Z=z) - P(Y=y|Z!=z)|
                # If this happens, instead, we have to use the estimation

                if (Y_eq_k_and_Z_eq_z == 0 and Y_eq_k_and_Z_not_eq_z != 0) or (Z_eq_z == 0 and Z_not_eq_z != 0):
                    denominator = 1 if z == 1 else 0
                    if average_probabilities and average_probabilities.get(f"{target}|{denominator}", None):
                        # In this case I use the estimation
                        violation_term = torch.abs(
                            average_probabilities[f"{target}|{denominator}"] - Y_eq_k_and_Z_not_eq_z / Z_not_eq_z
                        )
                    else:
                        # In this case instead I return 0, this only happens when
                        # we do not have the estimation (for instance in the first
                        # FL Round)
                        violation_term = torch.abs(Y_eq_k_and_Z_not_eq_z / Z_not_eq_z) - torch.abs(
                            Y_eq_k_and_Z_not_eq_z / Z_not_eq_z
                        )
                elif (Y_eq_k_and_Z_eq_z != 0 and Y_eq_k_and_Z_not_eq_z == 0) or (Z_not_eq_z == 0 and Z_eq_z != 0):
                    # This is just the other case
                    denominator = 1 if z == 0 else 0
                    if average_probabilities and average_probabilities.get(f"{target}|{denominator}", None):
                        violation_term = torch.abs(
                            (Y_eq_k_and_Z_eq_z / Z_eq_z) - average_probabilities[f"{target}|{denominator}"]
                        )
                    else:
                        violation_term = torch.abs(Y_eq_k_and_Z_eq_z / Z_eq_z) - torch.abs(Y_eq_k_and_Z_eq_z / Z_eq_z)

                else:
                    # In this case we have all the combinations,
                    # so we can compute the violation term using the classic formula
                    violation_term = torch.abs((Y_eq_k_and_Z_eq_z / Z_eq_z) - (Y_eq_k_and_Z_not_eq_z / Z_not_eq_z))

                fairness_violations.append(violation_term)

        fairness_violations_ = [item.item() if isinstance(item, torch.Tensor) else item for item in fairness_violations]

        # We get the index of the maximum violation term. Then we create a mask with
        # all zeros and we set to 1 the element at the index we found. We use this mask
        # to sum the violation terms and we return the result. This was needed because
        # when we started to work on this project we discovered that without this
        # some of the gradients were not computed correctly. I would not remove it
        # even if I'm not sure that it is needed anymore.
        index = fairness_violations_.index(max(fairness_violations_))
        fairness_violations = torch.stack(fairness_violations)
        mask = torch.full((fairness_violations.shape[0],), 0, dtype=torch.float32).to(device)
        mask[index] = 1
        res = torch.sum(mask * fairness_violations)

        if global_computation:
            if json_file:
                # This only works for binary case but we can extend it to a non binary case
                # if needed.
                for sens_value in json_file["possible_z"]:
                    poss_target = 0
                    try:
                        global_counters[sens_value] = (
                            global_counters[f"{poss_target}|{sens_value}"]
                            + global_counters[f"{abs(1 - poss_target)}|{sens_value}"]
                        )
                    except:
                        continue

                # the most stupid thing that I thought to remove the combinations that we do not
                # want to have. I'm removing this because later we will introduce noise
                # only on the data in the form "y|z" and not in the counters of the sensitive values.
                # Then we will derive the counters that we remove here from the counters
                # of the sensitive values and the combinations.
                for non_existing, _ in json_file["missing_combinations"]:
                    if non_existing in global_counters:
                        del global_counters[non_existing]

            return (res, global_counters)
        else:
            return res

    def violation_with_dataset(
        self,
        model: torch.nn.Module,
        dataset: torch.utils.data.DataLoader,
        average_probabilities: dict,
        device: torch.device,
        global_computation=False,
    ) -> torch.tensor:
        """
        When we want to compute the disparity metric on the entire dataset
        we can't directly use the forward function because we don't have the
        predictions and the sensitive attribute list for each batch.
        So in this function we just use the model to compute the predictions for
        all the samples in the dataset and we aggregate the results in a single
        final tensor that we pass to the forward function.
        This is used, for instance, to compute the disparity of the model
        on the test dataset.

        Args:
            model (torch.nn.Module): the model we want to evaluate
            dataset (torch.utils.data.DataLoader): the dataset we want to
                use during the evaluation
            average_probabilities (dict): in case of Federated learning, if a client
                has only a subset of the possible sensitive attributes, we can use the
                average probabilities of the other clients to estimate the probabilities
                of the missing sensitive attributes. This is None in centralised learning
            device (torch.device): the device we're using to train the model

        Returns:
            float: the disparity metric computed on the dataset
                passed as parameter
        """
        predictions = torch.tensor([]).to(device)
        sensitive_attribute_list = torch.tensor([]).to(device)
        targets = []
        model.eval()
        with torch.no_grad():
            for images, sensitive_attributes, target in dataset:
                images = images.to(device)
                target = target.to(device)

                output = model(images)

                predictions = torch.cat((predictions, output), 0)
                sensitive_attribute_list = torch.cat((sensitive_attribute_list, sensitive_attributes.to(device)), 0)
                targets += target.tolist()

        sensitive_attributes = list({item.item() for item in sensitive_attribute_list})
        target_list = list(set(targets))

        # now we just call the forward function with the "fake" predictions and the sensitive
        # attribute list we computed
        return self.forward(
            sensitive_attribute_list,
            device,
            predictions,
            sensitive_attributes,
            target_list,
            average_probabilities=average_probabilities,
            global_computation=global_computation,
        )

    def compute_violation_with_argmax(
        self,
        predictions_argmax: torch.tensor,
        sensitive_attribute_list: torch.tensor,
        current_target: int,
        current_sensitive_feature: int,
        weights: dict = None,
    ):
        """Debug function used to compute the DPL using the argmax function
        instead of the softmax.

        Args:
            predictions_argmax (torch.tensor): predictions of the model
            sensitive_attribute_list (torch.tensor): _description_
            target (int): The target we are considering
                in this iteration to compute the violation
            sensitive_feature (int): the sensitive feature
                we are considering in this iteration to
                compute the violation

        Returns:
            Tuple[int, int, int, int]: The number of times the
                prediction is equal to the target and the sensitive
                feature is equal to the sensitive feature we are
                considering in this iteration, the number of times
                the sensitive feature is equal to the sensitive
                feature we are considering in this iteration, the
                number of times the prediction is equal to the target
                and the sensitive feature is not equal to the sensitive
                feature we are considering in this iteration, the number
                of times the sensitive feature is not equal to the sensitive
                feature we are considering in this iteration
        """

        # opposite_sensitive_feature = 0 if current_sensitive_feature == 1 else 1

        # Z_eq_z_argmax = 0
        # Z_not_eq_z_argmax = 0
        # Y_eq_k_and_Z_eq_z_argmax = 0
        # Y_eq_k_and_Z_not_eq_z_argmax = 0

        # Z_eq_z and Z_not_eq_z are the denominators that we will use
        # in the DPL formula. |Z=z| and |Z!=z|

        Z_eq_z = len(sensitive_attribute_list[sensitive_attribute_list == current_sensitive_feature])
        Z_not_eq_z = len(sensitive_attribute_list[sensitive_attribute_list != current_sensitive_feature])
        Y_eq_k_and_Z_eq_z = len(
            predictions_argmax[
                (predictions_argmax == current_target) & (sensitive_attribute_list == current_sensitive_feature)
            ]
        )

        Y_eq_k_and_Z_not_eq_z = len(
            predictions_argmax[
                (predictions_argmax == current_target) & (sensitive_attribute_list != current_sensitive_feature)
            ]
        )

        if Z_eq_z == 0 and Z_not_eq_z != 0:
            return np.abs(Y_eq_k_and_Z_not_eq_z / Z_not_eq_z).item()
        elif Z_eq_z != 0 and Z_not_eq_z == 0:
            return np.abs(Y_eq_k_and_Z_eq_z / Z_eq_z).item()
        elif Z_eq_z == 0 and Z_not_eq_z == 0:
            return 0
        else:
            return np.abs(Y_eq_k_and_Z_eq_z / Z_eq_z - Y_eq_k_and_Z_not_eq_z / Z_not_eq_z).item()

    @staticmethod
    def compute_probabilities(
        predictions,
        sensitive_attribute_list,
        device: torch.device,
        possible_sensitive_attributes: list,
        possible_targets: list,
    ) -> torch.tensor:
        """This function computes the probabilities and the counters
            of each possible combination of target and sensitive attribute.
            It is used to compute the probabilities that we use to estimate
            the probabilities of the missing sensitive attributes in the
            Federated Learning scenario.

        Args:
            sensitive_attribute_list: a list with the value of
                the sensitive attribute for each sample in the batch
            device: the device we're using to train the model
            possible_targets: the possible target values we have in this
                dataset
            possible_sensitive_attributes: the possible values of the sensitive
                attribute

        Returns:
            (dict, dict): the probabilities and the counters of each possible combination
                of target and sensitive attribute
        """
        softmax_ = F.softmax(predictions, dim=1)

        # We compute the argmax of the predictions, this is used to count
        # the number of samples for each class that are predicted with one class
        # or with the other.
        predictions_argmax = torch.argmax(torch.tensor(predictions), dim=1).to(device)

        sensitive_attribute_list = torch.tensor([int(item) for item in sensitive_attribute_list])
        sensitive_attribute_list = sensitive_attribute_list.to(device)

        probabilities = {}
        counters = {}
        # possible_targets = [int(item) for item in possible_targets]
        possible_sensitive_attributes = [int(item) for item in possible_sensitive_attributes]

        for z in list(possible_sensitive_attributes):
            # if we are in a binary scenario we can just consider
            # one of the two values in the computation

            target = 1
            z = int(z)
            # Z_eq_z and Z_not_eq_z are the denominators that we will use
            # in the DPL formula. |Z=z| and |Z!=z|
            Z_eq_z = len(sensitive_attribute_list[sensitive_attribute_list == z])

            Y_eq_k_and_Z_eq_z = torch.sum(
                softmax_[(predictions_argmax == target) & (sensitive_attribute_list == z)][:, target]
            )

            Y_eq_k_and_Z_eq_z_argmax = len(
                predictions_argmax[(predictions_argmax == target) & (sensitive_attribute_list == z)]
            )

            probabilities[f"{target}|{z}"] = Y_eq_k_and_Z_eq_z
            probabilities[f"{z}"] = Z_eq_z
            counters[f"{target}|{z}"] = Y_eq_k_and_Z_eq_z_argmax

            counters[f"{z}"] = Z_eq_z

        return probabilities, counters

# Train the model

In [8]:
class LinearClassificationNet(nn.Module):
    """
    A fully-connected single-layer linear NN for classification.
    """

    def __init__(self, input_size, output_size):
        super(LinearClassificationNet, self).__init__()
        self.layer1 = nn.Linear(input_size, output_size, bias=False)

    def forward(self, x):
        x = self.layer1(x.float())
        return x

In [9]:
BATCH_SIZE = 256

train_loader = torch.utils.data.DataLoader(
    dutch_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

test_loader = torch.utils.data.DataLoader(
    dutch_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

In [10]:
class MixLoss(nn.Module):
    def __init__(self, weight_cel=1.0, weight_mse=1.0, reduction="mean"):
        super(MixLoss, self).__init__()
        self.cel_loss = nn.CrossEntropyLoss()
        self.cel_loss_2 = DisparityRegularizationLoss()
        self.weight_cel = weight_cel
        self.weight_mse = weight_mse
        self.reduction = reduction

    def forward(self, input, target):
        """
        Calculates a mixed loss combining NLLLoss and MSELoss.

        Args:
            input (torch.Tensor): The output from your model (e.g., m(input)).
                                  This will be used for both NLLLoss and MSELoss.
            target_nll (torch.Tensor): The target for NLLLoss (typically class labels).
            target_mse (torch.Tensor): The target for MSELoss (typically continuous values).

        Returns:
            torch.Tensor: The weighted sum of CELoss and MSELoss.
        """
        model_output = input[0]
        sensitive_value = input[1]
        loss_cel = self.cel_loss(model_output, target)
        loss_cel_2 = self.cel_loss_2(
            sensitive_attribute_list=sensitive_value,
            device="cpu",
            predictions=model_output,
            possible_sensitive_attributes=[0, 1],
            possible_targets=[0, 1],
        )  # You might need to adjust the input for MSE if it's not directly comparable to the NLL input.

        # return loss_cel

        lamda_value = 0.6
        total_loss = loss_cel + 1000 * loss_cel_2
        return total_loss

In [11]:
noise_multiplier = 0
max_grad_norm = 10000000000

In [12]:
seed_everything()
privacy_engine = PrivacyEngine()
LR = 0.01
# criterion = nn.CrossEntropyLoss()
criterion = MixLoss()
model = LinearClassificationNet(input_size=11, output_size=2)
optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0)

(
    model_gc,
    optimizer_gc,
    criterion_gc,
    train_loader,
) = privacy_engine.make_private(
    module=model,
    optimizer=optimizer,
    data_loader=train_loader,
    noise_multiplier=noise_multiplier,
    max_grad_norm=max_grad_norm,
    criterion=criterion,
    grad_sample_mode="ghost",
    poisson_sampling=False,
)

MAX_PHYSICAL_BATCH_SIZE = 1024

epochs = 10

with BatchMemoryManager(
    data_loader=train_loader,
    max_physical_batch_size=MAX_PHYSICAL_BATCH_SIZE,
    optimizer=optimizer_gc,
) as memory_safe_data_loader:
    for epoch in range(epochs):
        for batch_number, (input_data) in enumerate(memory_safe_data_loader, 0):
            input_data, sensitive_value, target_data = input_data[0], input_data[1], input_data[2]
            output_gc = model_gc(input_data)  # Forward pass
            optimizer_gc.zero_grad()
            loss = criterion_gc((output_gc, sensitive_value), target_data.long())
            loss.backward()
            optimizer_gc.step()  # Add noise and update the model
            accuracy = (output_gc.argmax(dim=1) == target_data.long()).float().mean().item()
            if batch_number % 100 == 0 and batch_number != 0:
                print(f"Batch {batch_number}, epoch {epoch}, Loss: {loss.item()}")
                print(f"Accuracy: {accuracy}")

                max_violation = 0
                for target in [0, 1]:
                    for sensitive_feature in [0, 1]:
                        violation = DisparityRegularizationLoss().compute_violation_with_argmax(
                            predictions_argmax=output_gc.argmax(dim=1),
                            sensitive_attribute_list=sensitive_value,
                            current_target=target,
                            current_sensitive_feature=sensitive_feature,
                        )
                        max_violation = max(max_violation, violation)
                print(f"Max Disparity: {max_violation}")

                for p in model.parameters():
                    if p.grad is not None:
                        print(f"Gradient norm: {p.grad.norm()}")

                print("----------------------")

/home/lcorbucci/PUFFLE/.venv/lib/python3.13/site-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
/tmp/ipykernel_1266313/3606101858.py:80: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  predictions_argmax = torch.argmax(torch.tensor(predictions), dim=1).to(device)


Batch 100, epoch 0, Loss: 2.4152114391326904
Accuracy: 0.52734375
Max Disparity: 0
Gradient norm: 3.2303714752197266
----------------------
Batch 100, epoch 1, Loss: 1.7197496891021729
Accuracy: 0.5625
Max Disparity: 0
Gradient norm: 9.929572105407715
----------------------
Batch 100, epoch 2, Loss: 2.6185903549194336
Accuracy: 0.48046875
Max Disparity: 0
Gradient norm: 9.862257957458496
----------------------
Batch 100, epoch 3, Loss: 1.9952571392059326
Accuracy: 0.53125
Max Disparity: 0
Gradient norm: 9.16888427734375
----------------------
Batch 100, epoch 4, Loss: 1.678085446357727
Accuracy: 0.5390625
Max Disparity: 0
Gradient norm: 10.478662490844727
----------------------
Batch 100, epoch 5, Loss: 1.704813838005066
Accuracy: 0.51171875
Max Disparity: 0
Gradient norm: 9.981842041015625
----------------------
Batch 100, epoch 6, Loss: 1.668477177619934
Accuracy: 0.5234375
Max Disparity: 0
Gradient norm: 9.61921501159668
----------------------
Batch 100, epoch 7, Loss: 1.57152068614

# NO GHOST

In [13]:
seed_everything()
privacy_engine = PrivacyEngine()
LR = 0.01
# criterion = nn.CrossEntropyLoss()
criterion = MixLoss()
model = LinearClassificationNet(input_size=11, output_size=2)
optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0)

(
    model_gc,
    optimizer_gc,
    train_loader,
) = privacy_engine.make_private(
    module=model,
    optimizer=optimizer,
    data_loader=train_loader,
    noise_multiplier=noise_multiplier,
    max_grad_norm=max_grad_norm,
    # criterion=criterion,
    grad_sample_mode="hooks",
    poisson_sampling=False,
)

MAX_PHYSICAL_BATCH_SIZE = 1024

epochs = 10

with BatchMemoryManager(
    data_loader=train_loader,
    max_physical_batch_size=MAX_PHYSICAL_BATCH_SIZE,
    optimizer=optimizer_gc,
) as memory_safe_data_loader:
    for epoch in range(epochs):
        for batch_number, (input_data) in enumerate(memory_safe_data_loader, 0):
            input_data, sensitive_value, target_data = input_data[0], input_data[1], input_data[2]
            output_gc = model_gc(input_data)  # Forward pass
            optimizer_gc.zero_grad()
            loss = criterion((output_gc, sensitive_value), target_data.long())

            # loss_1 = nn.CrossEntropyLoss()(output_gc, target_data.long())
            # loss_2 = DisparityRegularizationLoss()(sensitive_attribute_list=sensitive_value,
            #                                             device="cpu",
            #                                             predictions=output_gc,
            #                                             possible_sensitive_attributes=[0,1],
            #                                             possible_targets=[0,1])
            # loss = (1 - 0.6) * loss_1 + 0.6 * loss_2

            loss.backward()
            optimizer_gc.step()  # Add noise and update the model
            accuracy = (output_gc.argmax(dim=1) == target_data.long()).float().mean().item()
            if batch_number % 100 == 0 and batch_number != 0:
                print(f"Batch {batch_number}, epoch {epoch}, Loss: {loss.item()}")
                print(f"Accuracy: {accuracy}")

                max_violation = 0
                for target in [0, 1]:
                    for sensitive_feature in [0, 1]:
                        violation = DisparityRegularizationLoss().compute_violation_with_argmax(
                            predictions_argmax=output_gc.argmax(dim=1),
                            sensitive_attribute_list=sensitive_value,
                            current_target=target,
                            current_sensitive_feature=sensitive_feature,
                        )
                        max_violation = max(max_violation, violation)
                print(f"Max Disparity: {max_violation}")

                for p in model.parameters():
                    if p.grad is not None:
                        print(f"Gradient norm: {p.grad.norm()}")

                print("----------------------")

/home/lcorbucci/PUFFLE/.venv/lib/python3.13/site-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
/tmp/ipykernel_1266313/3606101858.py:80: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  predictions_argmax = torch.argmax(torch.tensor(predictions), dim=1).to(device)


Batch 100, epoch 0, Loss: 2.4152114391326904
Accuracy: 0.52734375
Max Disparity: 0
Gradient norm: 3.23037052154541
----------------------
Batch 100, epoch 1, Loss: 1.7197500467300415
Accuracy: 0.5625
Max Disparity: 0
Gradient norm: 9.9295654296875
----------------------
Batch 100, epoch 2, Loss: 1.6742810010910034
Accuracy: 0.48046875
Max Disparity: 0
Gradient norm: 8.750802993774414
----------------------
Batch 100, epoch 3, Loss: 1.5773695707321167
Accuracy: 0.53125
Max Disparity: 0
Gradient norm: 7.866127014160156
----------------------
Batch 100, epoch 4, Loss: 2.0638179779052734
Accuracy: 0.5390625
Max Disparity: 0
Gradient norm: 10.831135749816895
----------------------
Batch 100, epoch 5, Loss: 1.6935995817184448
Accuracy: 0.51171875
Max Disparity: 0
Gradient norm: 8.888175010681152
----------------------
Batch 100, epoch 6, Loss: 1.8003581762313843
Accuracy: 0.5234375
Max Disparity: 0
Gradient norm: 9.321377754211426
----------------------
Batch 100, epoch 7, Loss: 1.6571234464

In [ ]:
# # with ghost # NO SUM
# Batch 100, epoch 0, Loss: 0.6980053186416626
# Accuracy: 0.453125
# Max Disparity: 0.011937557392102893
# Gradient norm: 0.108265720307827

# # with ghost SUM
# Batch 100, epoch 0, Loss: 0.07164521515369415
# Accuracy: 0.47265625
# Max Disparity: 0
# Gradient norm: 0.05395718291401863

# # without ghost NO SUM
# Batch 100, epoch 0, Loss: 0.6980053186416626
# Accuracy: 0.453125
# Max Disparity: 0.011937557392102893
# Gradient norm: 0.1082657128572464

# # without ghost SUM
# Batch 100, epoch 0, Loss: 0.07164521515369415
# Accuracy: 0.47265625
# Max Disparity: 0
# Gradient norm: 0.05395718291401863